# Experiment 03: Evaluation & Character Error Analysis

This notebook performs:
1. **Full Evaluation**: Top-1 Accuracy, Top-5 Accuracy, Balanced Accuracy, and Macro F1.
2. **Confusion Matrix Heatmap**: Visualizing class predictions with Thai character labels.
3. **Top Confused Character Pairs**: Investigating visually ambiguous pairs (e.g. ด vs ต, ข vs ช, ผ vs พ, บ vs ป).
4. **Detailed Classification Report**: Per-class precision, recall, and F1-score breakdown.

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import torch

src_dir = (Path.cwd() / '..' / 'src').resolve()
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

from dataset import create_dataloaders
from transforms import get_val_transform
from models import build_model
from evaluate import evaluate_model_full, plot_confusion_matrix

DATASET_DIR = (Path.cwd() / '..' / '..' / '..' / 'ThaiCharacter Dataset' / 'round2').resolve()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

_, test_loader, _, test_df, class_to_idx = create_dataloaders(
    dataset_dir=DATASET_DIR,
    batch_size=64,
    target_size=(32, 32),
    test_transform=get_val_transform(),
    use_balanced_sampler=False,
)

print(f'Test Batches: {len(test_loader)}')


## 1. Load Trained Model & Run Inference

In [ ]:
model = build_model('custom_cnn', num_classes=len(class_to_idx))
ckpt_path = Path('../checkpoints/custom_cnn/best_model.pt')

if ckpt_path.exists():
    ckpt = torch.load(ckpt_path, map_location=device)
    model.load_state_dict(ckpt['model_state_dict'])
    print(f'Loaded checkpoint from {ckpt_path} (Epoch {ckpt.get("epoch", "N/A")})')
else:
    print('Checkpoint not found, evaluating initialized model for smoke test.')

results = evaluate_model_full(model, test_loader, class_to_idx, device=device)
print(f"\n🏆 Evaluation Summary:")
print(f"Top-1 Accuracy: {results['top1_accuracy']*100:.2f}%")
print(f"Top-5 Accuracy: {results['top5_accuracy']*100:.2f}%")
print(f"Macro F1-Score: {results['macro_f1']*100:.2f}%")
print(f"Weighted F1:   {results['weighted_f1']*100:.2f}%")


## 2. Confusion Matrix & Confused Character Pairs

In [ ]:
plot_confusion_matrix(results['confusion_matrix'], results['idx_to_char'], top_n=30)

conf_df = pd.DataFrame(results['top_confusions'])
print('\nTop 15 Most Confused Thai Character Pairs:')
display(conf_df[['true_char', 'pred_char', 'count']].head(15))
